# Lesson 03 - Agentic Design Patterns

ในบทเรียนนี้ เราจะสำรวจรูปแบบการออกแบบพื้นฐานสามอย่างสำหรับการสร้างเอเย่นต์ AI ที่มีประสิทธิภาพ:

1. **คำสั่งเอเย่นต์ที่ชัดเจน** — การสร้างพรอมต์ที่กำหนดบทบาทอย่างแม่นยำเพื่อชี้นำพฤติกรรมของเอเย่นต์
2. **ผลลัพธ์ที่มีโครงสร้างด้วยโมเดล Pydantic** — การทำให้แน่ใจว่าเอเย่นต์ส่งข้อมูลที่คาดเดาได้และผ่านการตรวจสอบแล้ว
3. **เอเย่นต์ที่รับผิดชอบหน้าที่เดียว** — การออกแบบเอเย่นต์ที่โฟกัสและทำหน้าที่ใดหน้าที่หนึ่งได้อย่างดี

เราจะใช้แต่ละรูปแบบกับสถานการณ์ **ผู้แนะนำจุดหมายปลายทางการท่องเที่ยว** โดยสร้างระบบอย่างเป็นขั้นตอนที่สามารถแนะนำจุดหมายปลายทาง ตรวจสอบความพร้อมใช้งาน และจัดการด้านโลจิสติกส์ได้


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## รูปแบบที่ 1: คำสั่งตัวแทนที่ชัดเจน

รูปแบบที่มีผลกระทบมากที่สุดก็เป็นรูปแบบที่ง่ายที่สุด: การเขียนคำสั่งที่ชัดเจนและละเอียดสำหรับตัวแทนของคุณ

คำสั่งที่ดีจะกำหนด:
- **ใคร** คือ ตัวแทน (บุคลิกและโทนเสียง)
- **อะไร** ที่ตัวแทนควรทำ (ความรับผิดชอบทีละขั้นตอน)
- **อย่างไร** ตัวแทนควรประพฤติ (ข้อจำกัดและสไตล์)

ด้านล่างนี้ เราสร้างตัวแทนผู้ช่วยวางแผนการเดินทางพร้อมคำสั่งที่ชัดเจนซึ่งกำหนดทุกคำตอบที่มันผลิตขึ้นมา


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output with Pydantic Models

ข้อความแบบอิสระเหมาะสำหรับการสนทนา แต่ระบบปลายทางต้องการข้อมูลที่มีโครงสร้าง  
การจับคู่ **โมเดล Pydantic** กับ **ฟังก์ชันเครื่องมือ** เราสามารถ:

- กำหนดสคีมาที่แน่นอนสำหรับผลลัพธ์ของเอเจนต์  
- ตรวจสอบความถูกต้องของการตอบกลับโดยอัตโนมัติ  
- รวมผลลัพธ์ของเอเจนต์เข้ากับตรรกะของแอปพลิเคชันได้อย่างน่าเชื่อถือ  

นอกจากนี้เรายังแนะนำเครื่องมือที่ส่งคืนรายละเอียดปลายทาง เพื่อให้เอเจนต์สามารถวางกรอบคำแนะนำบนข้อมูลจริงได้


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## รูปแบบที่ 3: ตัวแทนรับผิดชอบอย่างเดียว

งานที่ซับซ้อนจะได้ประโยชน์จากการแบ่งงานไปยังตัวแทนหลายตัวที่มีความมุ่งเน้นเฉพาะด้านโดยมีความรับผิดชอบอย่างเดียว:

- **ผู้เชี่ยวชาญด้านปลายทาง** ที่รู้เกี่ยวกับสถานที่และความพร้อมใช้งาน
- **นักวางแผนโลจิสติกส์** ที่ดูแลเที่ยวบิน โรงแรม และแผนการเดินทาง

นี่สะท้อนหลักการวิศวกรรมซอฟต์แวร์เรื่อง *การแยกความรับผิดชอบ* — ทำให้ตัวแทนแต่ละตัวทดสอบ ดูแล และปรับปรุงได้ง่ายขึ้นอย่างอิสระ


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## สรุป

ในบทเรียนนี้เราได้นำสามรูปแบบการออกแบบเอเย่นต์ไปใช้กับสถานการณ์แนะนำการเดินทาง:

| รูปแบบ | แนวคิดหลัก | ประโยชน์ |
|---|---|---|
| **คำแนะนำที่ชัดเจน** | กำหนดบุคลิกภาพ ความรับผิดชอบ และข้อจำกัดตั้งแต่ต้น | พฤติกรรมเอเย่นต์ที่สม่ำเสมอ และสอดคล้องกับแบรนด์ |
| **ผลลัพธ์ที่มีโครงสร้าง** | ใช้โมเดล Pydantic เป็นรูปแบบการตอบกลับ | ผลลัพธ์ที่ได้รับการตรวจสอบและอ่านได้โดยเครื่อง |
| **ความรับผิดชอบเดียว** | มอบงานที่เน้นเฉพาะให้แต่ละเอเย่นต์ | ง่ายต่อการทดสอบ ดูแลรักษา และประสานงาน |

รูปแบบเหล่านี้สามารถประสานกันได้อย่างเป็นธรรมชาติ — คุณสามารถรวมคำแนะนำที่ชัดเจนกับผลลัพธ์ที่มีโครงสร้างภายในเอเย่นต์ที่มีความรับผิดชอบเดียวเพื่อสร้างระบบที่มั่นคงและพร้อมใช้งานในผลิตภัณฑ์ได้


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
